# Lesson 2 · Exploring countries with Python

**Question:** What can country-level data tell us about life expectancy, income, and population?

This is an **optional exercise**. No programming experience is needed: run the code, change a few settings, and see what happens.

### Running cells
1. Click a code cell and press **Shift + Enter**. Results appear underneath.
2. Edit a cell and run it again to apply your changes.
3. Start at the top. When you change the settings in section 2, rerun **that settings cell, then each analysis cell you want to update**.

**Try this first:**


In [ ]:
print("Hello, world!")
print("I just ran my first Python cell.")

### Change something small

Change `NUMBER_OF_PEOPLE` to `6` and run the cell again.


In [ ]:
NUMBER_OF_PEOPLE = 4       # CHANGE ME: try 6 or 10
APPLES_PER_PERSON = 3        # CHANGE ME: try 2

total_apples = NUMBER_OF_PEOPLE * APPLES_PER_PERSON
print("Total apples:", total_apples)

---
## 1. Load the country data

The accompanying **`countries.csv`** contains one row per country per year. Each column records a different measurement. Run the cell to see the first 10 rows.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("countries.csv")
print("Rows:", len(data), "| Countries:", data["country"].nunique())
display(data.head(10))

### About the data

| Column | Meaning |
|:--|:--|
| `country` | Country name |
| `year` | Year of observation (2000–2024) |
| `region` | Geographic region |
| `life_expectancy` | Life expectancy at birth, in years |
| `gdp_per_capita` | GDP per person, PPP-adjusted, in **current** international dollars |
| `fertility` | Births per woman |
| `population` | Population estimate |

These are **real, historical World Bank World Development Indicators** for **141 countries over 2000–2024**, including Ukraine. Coverage and missing values vary by indicator. The figures are estimates and national averages, **not measurements of individuals**. GDP per capita is in *current* PPP international dollars, so changes across years should not be interpreted directly as inflation-adjusted economic growth.

**Think:** What might be missing that could help explain differences in life expectancy?


---
## 2. Choose what to explore

Change the values in the next cell, then run it. **To update a chart or summary, rerun its code cell too**; changing a setting does not automatically update earlier results.


In [ ]:
# ===== EDIT THESE SETTINGS =====
YEAR = 2024                     # Any year from 2000 to 2024
REGION = "All"                  # Or "Europe", "Asia", "Africa", "Americas", "Oceania"
X_VARIABLE = "gdp_per_capita"   # Or "population", "life_expectancy", "fertility"
Y_VARIABLE = "life_expectancy" # Or "gdp_per_capita", "population", "fertility"
LOG_X = True                    # Set False for a linear x-axis
# ===============================

# Check settings and choose the rows to study.
valid_years = sorted(data["year"].unique().tolist())
valid_regions = ["All"] + sorted(data["region"].dropna().unique().tolist())
valid_variables = ["life_expectancy", "gdp_per_capita", "population", "fertility"]
assert YEAR in valid_years, f"Choose YEAR from {min(valid_years)} to {max(valid_years)}"
assert REGION in valid_regions, f"Choose REGION from {valid_regions}"
assert X_VARIABLE in valid_variables, f"Choose X_VARIABLE from {valid_variables}"
assert Y_VARIABLE in valid_variables, f"Choose Y_VARIABLE from {valid_variables}"
assert isinstance(LOG_X, bool), "Set LOG_X to True or False (without quotes)"

selected = data[data["year"] == YEAR].copy()
if REGION != "All":
    selected = selected[selected["region"] == REGION].copy()

print(f"Selected {len(selected)} countries in {YEAR} | region: {REGION}")
display(selected[["country", "region", "life_expectancy", "gdp_per_capita", "population"]].head(8))


## 3. Summarize life expectancy

Compare the mean, median, and standard deviation. Are the mean and median close?


In [ ]:
values = selected["life_expectancy"].dropna()
print("Life expectancy in", YEAR, "(", REGION, ")")
print("Mean:", round(values.mean(), 1), "years")
print("Median:", round(values.median(), 1), "years")
print("Standard deviation:", round(values.std(), 1), "years")
print("Countries included:", len(values))

## 4. Plot the distribution

A histogram counts countries in each life-expectancy range. Is it symmetric? To compare years or regions, change and run **section 2's settings cell**, then rerun the histogram cell below.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(selected["life_expectancy"].dropna(), bins=12, edgecolor="white")
ax.set(xlabel="Life expectancy at birth (years)", ylabel="Number of countries",
       title=f"Life expectancy in {YEAR} — {REGION}")
plt.show()

## 5. Make a scatterplot

Each dot is a country. Try switching `LOG_X` between `True` and `False` in section 2, then rerunning that cell **and this scatterplot cell**. Which view is clearer?

Does a relationship in this chart prove causation?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for region, group in selected.groupby("region"):
    ax.scatter(group[X_VARIABLE], group[Y_VARIABLE], label=region, alpha=0.75, s=40)

if LOG_X:
    ax.set_xscale("log")
ax.set(xlabel=X_VARIABLE.replace("_", " ").title(),
       ylabel=Y_VARIABLE.replace("_", " ").title(),
       title=f"{Y_VARIABLE.replace('_',' ').title()} vs. {X_VARIABLE.replace('_',' ').title()} — {YEAR}")
if REGION == "All":
    ax.legend(title="Region", fontsize=9)
plt.show()

## 6. Follow one country over time

Change `COUNTRY` to compare life expectancy across 2000–2024.


In [ ]:
COUNTRY = "Ukraine"            # Try "Poland", "Japan", "Brazil", or "India"

assert COUNTRY in data["country"].values, "Choose a country from the dataset"
country_data = data[data["country"] == COUNTRY].sort_values("year")
display(country_data[["year", "life_expectancy", "gdp_per_capita"]])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(country_data["year"], country_data["life_expectancy"], marker="o", linewidth=2)
ax.set(xlabel="Year", ylabel="Life expectancy (years)",
       title=f"Life expectancy over time — {COUNTRY}")
ax.set_xticks(range(2000, 2025, 5))
plt.show()
